In [2]:
import os
import csv
import json
import sympy as sp
from IPython.display import Math,display

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
EQUATIONS = CONFIGS['experiments']['sr']['optimizedeqs']
MODELSDIR = CONFIGS['filepaths']['models']
with open('../data/splits/stats.json','r',encoding='utf-8') as f:
    STATS = json.load(f)
with open(os.path.join(MODELSDIR,'sr','optimized_equations.csv'),'r',encoding='utf-8') as f:
    REGISTRY = {row['name']:json.loads(row['constants']) for row in csv.DictReader(f)}

In [4]:
def standardize(variable,mean,std):
    return (variable-mean)/std

def parse_forms(equations,namespace):
    standardized = {}
    for name,equation in equations.items():
        standardized[name] = sp.sympify(equation['form'],locals={**namespace,**standardized})
    return standardized

def select_branch(expr,maxfn,index):
    return expr.replace(maxfn,lambda *args:args[index])

def check_equivalence(standardized,physical,sy,maxfn):
    residual = sy*standardized-physical
    return all(sp.simplify(select_branch(residual,maxfn,index))==0 for index in range(2))

def check_branch_order(standardized,physical,scale,maxfn):
    stdargs  = [arg for term in standardized.atoms(maxfn) for arg in term.args]
    physargs = [arg for term in physical.atoms(maxfn) for arg in term.args]
    return len(stdargs)==len(physargs) and all(sp.simplify(physarg-scale*stdarg)==0 for physarg,stdarg in zip(physargs,stdargs))

In [5]:
sy     = sp.Symbol('s_y',positive=True)
maxfn  = sp.Function('max')
lf     = sp.Symbol(r'\mathrm{LF}',real=True)
labels = {'bl':('B_L','B_L'),
          'rh':(r'\widehat{\mathrm{RH}}',r'\mathrm{RH}'),
          'thetae':(r'\widehat{\theta_e}',r'\theta_e'),
          'thetaestar':(r'\widehat{\theta_e^*}',r'\theta_e^*'),
          'shf':(r'\mathrm{SHF}',r'\mathrm{SHF}'),
          'lhf':(r'\mathrm{LHF}',r'\mathrm{LHF}')}
fields = {var:sp.Symbol(field,real=True) for var,(field,_) in labels.items()}
means  = {var:sp.Symbol(rf'\mu_{{{stat}}}',real=True) for var,(_,stat) in labels.items()}
stds   = {var:sp.Symbol(rf's_{{{stat}}}',positive=True) for var,(_,stat) in labels.items()}
c      = {f'c{i}':sp.Symbol(f'c_{{{i}}}',real=True) for i in range(1,15)}

namespace    = {**{var:standardize(fields[var],means[var],stds[var]) for var in labels},'lf':lf,'cube':lambda x:x**3,'max':maxfn,**c}
standardized = parse_forms(EQUATIONS,namespace)

In [6]:
lam,bc,beta,gamma,thetac,kappa,lfc,lamshf,lamlhf,lamthetae = sp.symbols(r'\lambda B_c \beta \gamma \Theta_c \kappa \mathrm{LF}_c \lambda_{\mathrm{SHF}} \lambda_{\mathrm{LHF}} \lambda_{\theta_e}',real=True)

moisture    = kappa*(fields['rh']-means['rh'])
instability = fields['thetae']-gamma*fields['thetaestar']-thetac
atm         = lam*maxfn(moisture,instability)**3
srall       = atm+(lfc-lf)**3*(lamthetae*(fields['thetae']-means['thetae'])+lamshf*(fields['shf']-means['shf']))-beta
physical    = {
    'sr_bl_eq':lam*(fields['bl']-bc)**3+beta,
    'sr_atm_eq':atm,
    'sr_sfc_eq':atm+lamshf*(lfc-lf)*(fields['shf']-means['shf'])+lamlhf*(fields['lhf']-means['lhf']),
    'sr_all_eq':srall,
    'sr_all_pc_eq':srall+lamthetae*(1-lfc)**3*(fields['thetae']-means['thetae'])}

atmconstants = {lam:sy*c['c3']/stds['thetae']**3,
                gamma:c['c4']*stds['thetae']/stds['thetaestar'],
                thetac:means['thetae']-c['c4']*stds['thetae']/stds['thetaestar']*means['thetaestar']+c['c5']*stds['thetae'],
                kappa:stds['thetae']/stds['rh']}
constants = {
    'sr_bl_eq':{lam:sy/stds['bl']**3,bc:means['bl']-c['c1']*stds['bl'],beta:sy*c['c2']},
    'sr_atm_eq':atmconstants,
    'sr_sfc_eq':{**atmconstants,lamshf:sy*c['c6']/stds['shf'],lfc:c['c7'],lamlhf:sy*c['c8']/stds['lhf']},
    'sr_all_eq':{**atmconstants,lamthetae:sy/stds['thetae'],lamshf:sy*c['c9']/stds['shf'],lfc:c['c10'],beta:sy*c['c11']},
    'sr_all_pc_eq':{**atmconstants,lamthetae:sy/stds['thetae'],lamshf:sy*c['c12']/stds['shf'],lfc:c['c13'],beta:sy*c['c14']}}

In [7]:
for name,equation in EQUATIONS.items():
    substituted = physical[name].subs(constants[name])
    equivalent  = check_equivalence(standardized[name],substituted,sy,maxfn)
    ordered     = check_branch_order(standardized[name],substituted,stds['thetae'],maxfn)
    shown       = {symbol:value for symbol,value in constants[name].items() if name in ('sr_bl_eq','sr_atm_eq') or symbol not in atmconstants}
    display(Math(rf'\textbf{{{equation["description"]}}} \quad z = {sp.latex(standardized[name])}'))
    display(Math(rf's_yz = {sp.latex(physical[name])}'))
    for symbol,value in shown.items():
        display(Math(rf'{sp.latex(symbol)} = {sp.latex(value)}'))
    print(f'{equation["description"]}: s_y*z equals the physical exponent on both max branches: {equivalent}, branches align: {ordered}')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

SR-ALL-PC: s_y*z equals the physical exponent on both max branches: True, branches align: True


In [8]:
features = {'PC1':fields['rh'],'PC2':fields['thetae'],'PC3':fields['thetaestar']}
branches = [r'M \geq I','I > M']
for name in ['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']:
    display(Math(rf'\textbf{{{EQUATIONS[name]["description"]}}}'))
    for pc,feature in features.items():
        for index,branch in enumerate(branches):
            derivative = sp.diff(select_branch(physical[name],maxfn,index),feature)
            display(Math(rf'\mathrm{{{pc}}},\ {branch}: \quad \partial(s_yz)/\partial {sp.latex(feature)} = {sp.latex(derivative)}'))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
a,b         = lfc-lf,1-lfc
coefficient = sp.diff(select_branch(physical['sr_all_pc_eq'],maxfn,0),fields['thetae'])
display(Math(rf'\partial(s_yz)/\partial {sp.latex(fields["thetae"])}\big|_{{M \geq I}} = {sp.latex(coefficient)}'))
print(f'Coefficient equals lambda_thetae*(a^3+b^3): {sp.simplify(coefficient-lamthetae*(a**3+b**3))==0}')
print(f'a^3+b^3 equals (1-LF)(a^2-ab+b^2): {sp.expand(a**3+b**3-(1-lf)*(a**2-a*b+b**2))==0}')
print(f'a^2-ab+b^2 equals (a-b/2)^2+3b^2/4: {sp.expand(a**2-a*b+b**2-((a-b/2)**2+sp.Rational(3,4)*b**2))==0}')

<IPython.core.display.Math object>

Coefficient equals lambda_thetae*(a^3+b^3): True
a^3+b^3 equals (1-LF)(a^2-ab+b^2): True
a^2-ab+b^2 equals (a-b/2)^2+3b^2/4: True


In [10]:
values = {sy:STATS['tp_std'],
          **{means[var]:STATS[f'{var}_mean'] for var in labels},
          **{stds[var]:STATS[f'{var}_std'] for var in labels},
          **{c[cname]:cvalue for fitted in REGISTRY.values() for cname,cvalue in fitted.items()}}

for name,equation in EQUATIONS.items():
    shown = {symbol:value for symbol,value in constants[name].items() if name in ('sr_bl_eq','sr_atm_eq') or symbol not in atmconstants}
    for symbol,value in shown.items():
        print(f'{equation["description"]}: {sp.latex(symbol)} = {float(value.subs(values)):.4g}')
for var in ['bl','rh','thetae','thetaestar','shf','lhf']:
    print(f'Training mean: {sp.latex(means[var])} = {STATS[f"{var}_mean"]:.4g}')
print(f's_y = {STATS["tp_std"]:.4g}')

SR-BL: \lambda = 1660
SR-BL: B_{c} = -0.09647
SR-BL: \beta = 0.07398
SR-ATM: \lambda = 0.001106
SR-ATM: \gamma = 1.058
SR-ATM: \Theta_{c} = -30.45
SR-ATM: \kappa = 0.4137
SR-SFC: \lambda_{\mathrm{SHF}} = 0.0125
SR-SFC: \mathrm{LF}_c = 0.74
SR-SFC: \lambda_{\mathrm{LHF}} = 0.001564
SR-ALL: \lambda_{\theta_e} = 0.05816
SR-ALL: \lambda_{\mathrm{SHF}} = 0.06552
SR-ALL: \mathrm{LF}_c = 0.73
SR-ALL: \beta = 0.06341
SR-ALL-PC: \lambda_{\theta_e} = 0.05816
SR-ALL-PC: \lambda_{\mathrm{SHF}} = 0.06515
SR-ALL-PC: \mathrm{LF}_c = 0.73
SR-ALL-PC: \beta = 0.07398
Training mean: \mu_{B_L} = -0.07599
Training mean: \mu_{\mathrm{RH}} = 68.07
Training mean: \mu_{\theta_e} = 342.4
Training mean: \mu_{\theta_e^*} = 355.5
Training mean: \mu_{\mathrm{SHF}} = 13.91
Training mean: \mu_{\mathrm{LHF}} = 124.4
s_y = 0.5284
